# EPCLM-004 M1 — Unified Data & E0
Enable Internet + GPU T4 and existing GITHUB_TOKEN secret. Save & Run All. CPU tiny → E0 MQAR 32 updates → E0 WikiText 4 updates → existing publisher. No Screening/Confirmation. Tokenizer is trained only on locked WikiText train data; test remains sealed.

In [ ]:
from pathlib import Path
import os, signal, subprocess, sys, time

started = time.monotonic()
deadline = started + 7200
branch = 'research/epclm-004-m1-e0'
repo = Path('/kaggle/working/minicells-epclm004-m1')
assets = Path('/kaggle/working/epclm004-m1-assets')

def run(command, cwd=None, limit=1800):
    timeout = min(limit, deadline-time.monotonic())
    if timeout <= 0: raise TimeoutError('M1 smoke budget exhausted')
    process = subprocess.Popen(command,cwd=cwd,start_new_session=True)
    try:
        code = process.wait(timeout=timeout)
    except subprocess.TimeoutExpired:
        os.killpg(process.pid,signal.SIGTERM)
        try: process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid,signal.SIGKILL); process.wait()
        raise TimeoutError('Output retained under /kaggle/working; no automatic retraining')
    if code: raise RuntimeError('M1 stopped; inspect saved decision.json in /kaggle/working')

if not repo.exists():
    run(['git','clone','--depth=1','--single-branch','--branch',branch,'https://github.com/ArcheLabs/mini-cells.git',str(repo)],limit=180)
else:
    run(['git','-C',str(repo),'fetch','--depth=1','origin',branch],limit=180)
    run(['git','-C',str(repo),'switch','--detach','FETCH_HEAD'],limit=60)
source = subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()
base = repo/'research/projects/cellular-language-models/experiments/epclm-004-unified-memory-10m'
run([sys.executable,'-m','pip','install','-q','-r',str(base/'requirements-m1.txt')],limit=300)
os.environ['OMP_NUM_THREADS']='1'
os.environ['PYTEST_DISABLE_PLUGIN_AUTOLOAD']='1'
run([sys.executable,'-m','pytest','-q',str(base/'tests')],cwd=repo,limit=180)
# CPU tiny uses the exact same data, supervision and optimizer path as E0 GPU.
run([sys.executable,str(base/'run.py'),'--profile','cpu-tiny','--task','mqar',
     '--assets',str(assets),'--output',f'/kaggle/working/epclm004-m1-cpu-{source[:12]}','--publish'],cwd=repo,limit=1800)
for task in ('mqar','lm'):
    # Publisher descends from main and switches branches. Restore the recorded
    # source before the next task; assets/checkpoints live outside the checkout.
    run(['git','-C',str(repo),'switch','--detach',source],limit=60)
    output = Path(f'/kaggle/working/epclm004-m1-gpu-{task}-{source[:12]}')
    run([sys.executable,str(base/'run.py'),'--profile','gpu-smoke','--device','cuda',
         '--task',task,'--assets',str(assets),'--output',str(output),'--publish'],cwd=repo,limit=2100)
